In [1]:
# Installation of required libraries
# You need pip version compatiable with your Python version

# In case if you need to install missing libraries,
# Create a cell, and write "!pip install library_name"3
#!pip install jupyter_server
# !pip install PyMuPDF
# !pip install openai
# !pip install tiktoken
# !pip install anthropic
#!pip install google-generativeai

In [2]:
# import the required libraries
import fitz
import tiktoken

import numpy as np
import pandas as pd

import openai
from openai import OpenAI
from dotenv import dotenv_values
import anthropic
import base64
import requests
import math, random
import os, time, sys
import re, string

### Import modules
from pdf_extracter.image_module import *
from pdf_extracter.text_module import *
from post_processing import *
from cost_calculator import *
from extract_using_llm import *
from prompt_design import *

/Users/naman/miniconda3/envs/mayo/lib/python3.12/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [ ]:
# dis = verify_responses(responses_path="/Users/naman/Desktop/ASU/Mayo Clinic/LLMs_ExtractionWithCrossCritique-main/LLM_Extraction_and_CrossCritique/Code/gpt-4o-mini_PP__00:03:47.txt"
#                  ,document_path="..//trial_folder/NCT02446405_Sweeney_ENZAMET_Lancet Onc23.pdf")

### Extraction

In [ ]:
### For Jupyter Notebook Execution ###
choice = input("Enter the Choice for LLMs Execution (Y/N): ")
if choice == 'Y' or choice == 'y':
    ### INPUT ###
    pdf_folder = "../trial_folder" # "/kaggle/input/extractiontrain"
    variable_file = "sysReviewColumns - Definitions.csv" #"/kaggle/input/pdftest/variables.csv"
    gemini_model =  "gemini-1.5-flash"# claude-3-opus-20240229 / gpt-4-0125-preview --- For testing, use gpt-3.5-turbo
    gpt_model = "gpt-4o-mini"
    prompt_size = 10
    print ("\n")
    ### LLM_CALL ###
    extract_using_llms(gemini_model, prompt_size, pdf_folder, variable_file)

else:
    # reading recorded responses
    raw_resp_file = "/Users/naman/Desktop/ASU/Mayo Clinic/LLMs_ExtractionWithCrossCritique-main/LLM_Extraction_and_CrossCritique/Code/gemini-1.5-flash__15:20:21.txt" #"/kaggle/input/pdftest/gpt-4-0125-preview__19_04_31.txt"
    pp_resp_file = "./gemini-1.5-flash__15/10/2024:15:23" #"/kaggle/input/pdftest/gpt-4-0125-preview_PP__19_04_38.txt"
    extract_from_files(raw_resp_file, pp_resp_file)

### Dis-Agreement Resolution

In [ ]:
# from disagreement_resolution import *

# a = input("Do you to execute Disagreement Resolution Module? Y/N")
# if a == "Y" or a == "y":
#     model = input("Enter the Model Name") # or claude-3-opus-20240229 or gpt-4-0125-preview
#     key = input("Enter the Key")
#     test_folder = input("Enter the Path to the PDF Folder") # "/kaggle/input/extractiontrain"
#     agreement_filepath = input("Enter the Path to the Annotated Agreement Matching File") # "/kaggle/input/pdftest/TRAIN_GPT_Claude_agreement.xlsx"    
#     disagreement_resolution(test_folder, model, model_key, agreement_filepath)
# else:
#     print ("You Choose Not to Execute Agreement/Disagreement Module.")

* ***Graphical representation of the Results***

In [ ]:
# # data from https://allisonhorst.github.io/palmerpenguins/

# import matplotlib.pyplot as plt
# import numpy as np

# species = ("Baseline Prompts", "Eng. Prompts")
# penguin_means = {
#     'GPT-3.5': (55, 60),
#     'GPT-4': (62, 68),
# }

# x = np.arange(len(species))  # the label locations
# width = 0.25  # the width of the bars
# multiplier = 0

# fig, ax = plt.subplots(layout='constrained')

# for attribute, measurement in penguin_means.items():
#     offset = width * multiplier
#     rects = ax.bar(x + offset, measurement, width, label=attribute)
#     ax.bar_label(rects, padding=2)
#     multiplier += 1

# # Add some text for labels, title and custom x-axis tick labels, etc.
# ax.set_ylabel('Performance')
# ax.set_title('Performance Comparison of GPT Models')
# ax.set_xticks(x + width - 0.13, species)
# ax.legend(loc='upper left', ncols=2)
# ax.set_ylim(0, 100)

# plt.show()

* ***Random Sampling***

In [ ]:
# filenames = os.listdir("/kaggle/input/extraction/") # get all filenames from the folder
# print ("Number of Files:", len(filenames)) # verify that you have extracted all the files

# files = []
# for file in filenames:
#     if ".pdf" in file and "SLIDE" not in file:
#         files.append(file)
        
# print (len(files))

# training_files = random.sample(files, 5)
# for file in training_files:
#     print (file)

In [4]:
df = pd.read_excel("var_definitions.xlsx",sheet_name='Defs_with_labels')
df

,Column Name,Definition,Reasoning Path,Label
0,NCT,"National Clinical Trial identifier, a unique i...",NaN,NCT
1,PubMed ID,A unique identifier assigned to a clinical tri...,NaN,PubMed ID
2,Trial Name,The title or identifier of the clinical trial ...,NaN,Trial Name
3,Author,The name of the person or people responsible f...,NaN,Author
4,Year,The year in which the clinical trial was publi...,NaN,Year
...,...,...,...,...
128,Median PFS (mo) | Low volume | Control,Median Progression-Free Survival duration in m...,NaN,Median PFS (mo)
129,COE_RCT_IND_OVERALL_RJ,Overall risk judgment of individual criteria o...,NaN,COE_RCT_IND_OVERALL_RJ
130,Add-on Treatment,Name of additional therapy or intervention giv...,NaN,Add-on Treatment
131,Treatment Class,A categorization or classification of the trea...,NaN,Treatment Class


In [6]:
unique_labels = df['Label'].unique()
from model_inference.gemini import ask_gemini
label_instruct = {}
all_inputs = []
for label in unique_labels:
    temp = df.loc[df['Label'] == label]
    input  = "\n".join(f"{col} : {defn}" for col,defn in zip(temp['Column Name'],temp['Definition']))
    instructions = ask_gemini(text = input,prompt_path="prompts/gen_instruct.txt",key=3,model_name="gemini-2.0-flash-exp")
    with open(f"instructions/{label}.txt","w") as f:
        f.write(instructions)
    time.sleep(10)
    label_instruct[label] = instructions
    

FileNotFoundError: [Errno 2] No such file or directory: 'instructions/Original/Follow Up.txt'

In [6]:
import json
with open("instructions.json", "w") as json_file:
    json.dump(label_instruct, json_file, indent=4)

In [9]:
all_inputs

['NCT : National Clinical Trial identifier, a unique identification number assigned to a clinical trial.',
 'PubMed ID : A unique identifier assigned to a clinical trial publication in the PubMed database.',
 'Trial Name : The title or identifier of the clinical trial being referred to in the study.',
 'Author : The name of the person or people responsible for writing or publishing the clinical trial study.',
 'Year : The year in which the clinical trial was published.',
 'Full Pub or Abstract : Indicates whether the study results were fully published or only an abstract is available.',
 'Phase : The stage or step in the clinical trial process, typically denoted by a Roman numeral (e.g. I, II, III, IV), which defines the scope, objectives, and duration of the trial.',
 'Original/Follow Up : The designation of the clinical trial as either an original study or a follow-up study.',
 'Number of Arms Included : The total count of experimental treatment arms and control arms included in the 